# Chapter 3 — Description Logics
### Notebook 4 · Exercises

*Book reference: Section 3.4*

The book's exercises, executable. Assertions are the marking scheme.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch03_toolkit as dl
import pandas as pd
A = dl.Atomic
logging.getLogger("dspy").setLevel(logging.WARNING)

### Exercise R1 — Translate English into DL

Express each statement as a DL axiom and verify the intended entailment holds.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
t = dl.TBox()
# 'A vegetarian is a person that eats no meat.'
t.add(A('Vegetarian'), dl.And(A('Person'), dl.ForAll('eats', dl.Not(A('Meat')))), True)
# 'A vegan is a vegetarian that eats no dairy.'
t.add(A('Vegan'), dl.And(A('Vegetarian'), dl.ForAll('eats', dl.Not(A('Dairy')))), True)
# 'Steak is meat.'
t.add(A('Steak'), A('Meat'))

print('Vegan <= Vegetarian      :', dl.subsumes(A('Vegan'), A('Vegetarian'), t))
print('Vegetarian <= Vegan      :', dl.subsumes(A('Vegetarian'), A('Vegan'), t))
print('Vegetarian eating steak? :',
      dl.satisfiable(dl.And(A('Vegetarian'), dl.Exists('eats', A('Steak'))), t).satisfiable)
assert dl.subsumes(A('Vegan'), A('Vegetarian'), t)
assert not dl.subsumes(A('Vegetarian'), A('Vegan'), t)
assert not dl.satisfiable(dl.And(A('Vegetarian'),
                                 dl.Exists('eats', A('Steak'))), t).satisfiable
print('\nSubsumption runs one way only -- the same asymmetry as Chapter 2\'s\n'
      'quantifier order, and just as easy to assert backwards.')

### Exercise R2 — Name the logic for five knowledge bases

Report the DL for each, and rank them by worst-case reasoning complexity.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
builders = {
    'plain':      lambda t: t.add(A('A'), A('B')),
    'negation':   lambda t: t.add(A('A'), dl.Not(A('B'))),
    'transitive': lambda t: (t.add(A('A'), dl.Exists('r', A('B'))),
                             t.transitive_roles.add('r')),
    'inverse+num': lambda t: (t.add(A('A'), dl.Exists(dl.Inverse('r'), A('B'))),
                              t.add(A('B'), dl.AtLeast(2, 'r', A('C')))),
    'everything': lambda t: (t.add(A('A'), dl.Exists(dl.Inverse('r'), A('B'))),
                             t.add(A('B'), dl.AtLeast(2, 'r', A('C'))),
                             t.transitive_roles.add('r'),
                             t.role_hierarchy.append(('r', 's')),
                             t.nominals.add('bob')),
}
rows = []
for name, build in builders.items():
    t = dl.TBox(); build(t)
    rows.append({'kb': name, 'dl': dl.dl_name(t)})
import pandas as pd; print(pd.DataFrame(rows).to_string(index=False))
assert dl.dl_name(_t1 := (lambda: (t1 := dl.TBox(), builders['transitive'](t1))[0])()) == 'S'
print('\nRanking by worst case: ALC = S = SHIQ (ExpTime) < SHOIQ (NExpTime).\n'
      'Note ALC and SHIQ share a complexity class -- expressivity and complexity\n'
      'do not increase in lockstep, which is exactly why the letters are worth\n'
      'knowing rather than guessing.')

### Exercise R3 — Explain an unsatisfiability to a colleague

Take the unsatisfiable `Giraffe ⊓ Carnivore`, and produce the shortest chain of axioms that explains it — the *justification*, in DL terms.

> **Hint.** Search subsets of the axioms from smallest upward, keeping the first that still makes the concept unsatisfiable.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
w = dl.wildlife_tbox()
target = dl.And(A('Giraffe'), A('Carnivore'))
assert not dl.satisfiable(target, w).satisfiable

# A justification is a minimal subset of axioms that still entails the problem.
import itertools
axioms = w.axioms
justification = None
for size in range(1, len(axioms) + 1):
    for subset in itertools.combinations(axioms, size):
        t = dl.TBox(list(subset))
        if not dl.satisfiable(target, t).satisfiable:
            justification = subset
            break
    if justification:
        break

print(f'minimal justification ({len(justification)} of {len(axioms)} axioms):')
for ax in justification:
    print('  ', ax)
assert justification and len(justification) < len(axioms)
print('\nThis is what a debugging tool should show you -- not "unsatisfiable",\n'
      'but the smallest set of axioms you must change. Computing it by brute\n'
      'force over subsets is exponential; real tools are cleverer, but the\n'
      'definition is exactly this.')

## Where this leaves you

You have a reasoner, you can name the logic you are in, and you can explain an unsatisfiability with a minimal justification. Notebook 5 asks the question a practising engineer actually faces: reasoning is **sound but expensive** — when is it worth calling?